# Fil rouge : boîte à outils

Ce notebook **fonctionne tel quel** : il entraîne un ResNet18 et mesure tout ce qu'il faut. À
vous de le modifier pour explorer la piste que vous avez choisie : <https://bonben.github.io/ensta-tp-deep-learning/04_fil_rouge/>.

Sur Colab : *Exécution > Modifier le type d'exécution > GPU T4*.

## Les données

In [2]:
import os
import tarfile
import urllib.request

CIFAR_URL = "https://github.com/bonben/ensta-tp-deep-learning/releases/download/cifar10/cifar-10-python.tar.gz"

if not os.path.exists("data/cifar-10-batches-py"):
    urllib.request.urlretrieve(CIFAR_URL, "cifar.tar.gz")
    with tarfile.open("cifar.tar.gz") as tar:
        tar.extractall("data", filter="data")
    os.remove("cifar.tar.gz")
print("CIFAR-10 est prêt")

CIFAR-10 est prêt


## Les outils

- `count_params(model)` : nombre de paramètres appris ;
- `count_macs(model)` : multiplications-additions pour classer **une** image ;
- `fit(model, epochs, ...)` : entraîne et évalue. **Tous les réglages sont des arguments**,
  avec la recette de la baseline par défaut. À la fin, il affiche un résumé
  de l'essai.

In [3]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.transforms import v2
from torch.utils.flop_counter import FlopCounterMode
import matplotlib.pyplot as plt

torch.multiprocessing.set_start_method("fork", force=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Calcul sur :", device)

MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
NORMALIZE = [v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)]

# L'augmentation de la baseline (cours 2). Pour en changer, créez votre propre Compose.
transform_train = v2.Compose([v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip()] + NORMALIZE)
transform_val = v2.Compose(NORMALIZE)
val_set = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=transform_val)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=256, shuffle=False, num_workers=2)


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def count_macs(model):
    model.eval()
    with FlopCounterMode(display=False) as counter:
        model(torch.zeros(1, 3, 32, 32, device=next(model.parameters()).device))
    return counter.get_total_flops() // 2


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            correct += (model(inputs).argmax(1) == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total


def fit(model, epochs, lr=0.1, weight_decay=5e-4, batch_size=128, label_smoothing=0.0,
        transform=transform_train, seed=0, name="essai"):
    """Entraîne avec SGD (momentum 0.9) et un scheduler cosinus. Renvoie l'historique."""
    torch.manual_seed(seed)
    train_set = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=transform)
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {"train_acc": [], "val_acc": []}
    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        correct, total = 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        scheduler.step()
        history["train_acc"].append(100 * correct / total)
        history["val_acc"].append(evaluate(model, val_loader))
        print(f"époque {epoch + 1}/{epochs} | train {history['train_acc'][-1]:.1f} % | "
              f"val {history['val_acc'][-1]:.1f} % | {time.time() - t0:.0f} s")
    print(f"\nRÉSUMÉ : {name} | {count_params(model)} paramètres | "
          f"{count_macs(model)} MAC | {epochs} époques | graine {seed} | "
          f"val au dernier epoch {history['val_acc'][-1]:.2f} %")
    return history


def plot(*histories, labels=None):
    """Superpose les courbes de validation de plusieurs essais."""
    for i, h in enumerate(histories):
        plt.plot(h["val_acc"], label=labels[i] if labels else f"essai {i + 1}")
    plt.xlabel("époque"); plt.ylabel("précision de validation (%)")
    plt.grid(alpha=0.3); plt.legend(); plt.show()

Calcul sur : cuda


## Le réseau

Le code du ResNet est **ici, dans le notebook** (c'est le même que `resnet.py`). Pour
l'explorer, modifiez cette cellule ou écrivez vos propres classes en dessous, **réexécutez la
cellule**, puis **recréez le modèle** : un modèle déjà créé garde l'ancien code.

In [4]:
class BasicBlock(nn.Module):
    """Deux convolutions 3×3 et un raccourci (la connexion résiduelle)."""

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        # Si la taille change (stride ou nombre de FM), le raccourci est une convolution 1×1.
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)  # la connexion résiduelle
        return F.relu(out)


class ResNet(nn.Module):
    """4 étages ; le nombre de FM vaut width, 2·width, 4·width, 8·width."""

    def __init__(self, block, num_blocks, num_classes=10, width=64):
        super().__init__()
        self.in_planes = width
        self.conv1 = nn.Conv2d(3, width, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(width)
        self.layer1 = self._make_layer(block, width, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 2 * width, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 4 * width, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 8 * width, num_blocks[3], stride=2)
        self.linear = nn.Linear(8 * width, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        layers = []
        for s in [stride] + [1] * (num_blocks - 1):
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))  # 32×32
        out = self.layer1(out)                 # 32×32
        out = self.layer2(out)                 # 16×16
        out = self.layer3(out)                 # 8×8
        out = self.layer4(out)                 # 4×4
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.linear(out)

## Un premier essai

Le point de départ : votre baseline, un ResNet18. Changez le modèle et les arguments de `fit`
selon votre piste. **Affichez toujours paramètres et MAC avant d'entraîner** : c'est gratuit,
et ça évite d'attendre 10 minutes pour un réseau trop gros.

In [5]:
model = ResNet(BasicBlock, [2, 2, 2, 2], width=64)
print(f"{count_params(model):,} paramètres, {count_macs(model.to(device)):,} MAC")

11,173,962 paramètres, 555,422,720 MAC


In [ ]:
h_base = fit(model, epochs=15, name="ResNet18 baseline")

époque 1/15 | train 26.8 % | val 38.2 % | 54 s
époque 2/15 | train 43.7 % | val 48.2 % | 57 s
époque 3/15 | train 54.1 % | val 50.8 % | 58 s
époque 4/15 | train 62.4 % | val 66.0 % | 57 s
époque 5/15 | train 68.1 % | val 69.2 % | 57 s
époque 6/15 | train 73.6 % | val 74.3 % | 57 s
époque 7/15 | train 77.1 % | val 76.7 % | 57 s
époque 8/15 | train 80.7 % | val 77.1 % | 58 s
époque 9/15 | train 83.2 % | val 79.5 % | 58 s
époque 10/15 | train 85.0 % | val 84.1 % | 57 s
époque 11/15 | train 87.3 % | val 86.2 % | 57 s
époque 12/15 | train 89.1 % | val 86.1 % | 57 s
époque 13/15 | train 91.1 % | val 88.1 % | 58 s


## Vos essais

Une cellule par essai, avec un nom qui dit ce que vous avez changé. Comparez avec `plot` :

```python
plot(h_base, h_essai1, labels=["baseline", "essai 1"])
```

In [ ]:
# model = ...
# h_essai1 = fit(model, epochs=15, name="...")

In [ ]:
# Augmentation des étages par bloc (ResNet34 : [3, 4, 6, 3]) et augmentation des époques pour atteindre une précision élevée
# Note : Pensez à bien exécuter la cellule contenant la définition de ResNet avant de lancer celle-ci !
model_deep = ResNet(BasicBlock, [3, 4, 6, 3], width=64)
print(f"{count_params(model_deep):,} paramètres, {count_macs(model_deep.to(device)):,} MAC")

# Entraînement avec 30 époques (ou plus) pour maximiser l'accuracy
h_deep = fit(model_deep, epochs=30, lr=0.1, name="ResNet34_deep_epochs")